# Looking at the plots from ``Unlearn-Simple/MUSE/notebooks/eval_with_ILL_raw_files.ipynb``

I will describe the interesting things:


# Summary Report

## Overview of Experimental Design and Report Structure

This report summarizes key insights and findings from a series of experiments conducted in the notebook `Unlearn-Simple/MUSE/notebooks/eval_with_ILL_raw_files.ipynb`.

The experiments leverage the two-phase framework—neighborhood generation (pre-run) and ILL feature extraction (post-run)—to analyze and distinguish between the **retain**, **forget**, and **holdout** data subsets in the context of machine unlearning.

In this report, we present a detailed exploration of how various ILL-derived metrics—such as loss, variance, gradient, and neighborhood-based features—can reveal new phenomena and structural differences between these subsets.

The report is organized as follows:
- **Descriptive Analysis:** We begin with histograms and distributional analyses of key ILL features, highlighting how each subset differs in terms of loss, neighborhood structure, and landscape properties.
- **Discriminative Power:** We assess which features are most effective for distinguishing between the subsets, using classification models and feature importance metrics.
- **Visualization:** We employ dimensionality reduction techniques (PCA, t-SNE, UMAP, Isomap) to visualize the separability and structure of the data in the ILL feature space.
- **Robustness and Transfer:** We analyze confusion matrices, ROC curves, and cross-dataset prediction transfer to evaluate the robustness of the features and the challenges in isolating the forget subset without harming retain or holdout performance.
- **Neighborhood Insights:** We discuss the impact of our neighborhood generation strategy and how it enhances the interpretability of local loss landscapes.
- **Implications for Unlearning:** Finally, we synthesize these findings to provide actionable insights and recommendations for future unlearning algorithm development and evaluation.

Through these analyses, the report aims to demonstrate the value and flexibility of the two-phase approach for both benchmarking and deeper understanding

# Reminder for our research goals and motivations
## Introduction

This notebook introduces a new two-phase framework for analyzing machine unlearning, with a primary focus on the methods themselves and their advantages for deeper model analysis and classification. The two phases are:

- **Pre-run Phase: Neighborhood Generation**  
  This phase presents a novel method for generating sentence neighborhoods, designed to reflect the local structure of the model’s feature space. Unlike standard data augmentation or random sampling, this approach enables more meaningful and context-aware neighborhood construction, which can be used independently for post-run evaluation or in combination with other analysis tools.

- **Post-run Phase: Input Loss Landscape (ILL) Feature Extraction**  
  In this phase, we introduce a systematic approach for extracting a rich set of ILL features—including loss, variance, gradient, and landscape-based metrics—from the model. This method allows for detailed characterization of data points and their neighborhoods, and can be applied with any neighborhood generation strategy.

These phases are modular: each can be used independently or together, offering flexibility for different research and evaluation needs. When combined, they provide a powerful toolkit for distinguishing between three critical data subsets in unlearning:

- **Forget Subset:** Data the model is explicitly instructed to "unlearn."
- **Retain Subset:** Data the model should "remember," ensuring unlearning does not degrade overall performance.
- **Holdout Subset:** Data never seen during training or unlearning, serving as an independent benchmark.

While the primary goal of this research is to introduce and benchmark these two phases, their combination also serves as an investigative tool that may reveal new phenomena and insights about model behavior and the effects of unlearning. If such insights are uncovered, the focus of the research can naturally shift toward using these methods for deeper understanding and discovery.

**Key Components and Contributions:**

1. **Pre-run Phase: Neighborhood Generation**  
   A new, context-aware method for constructing sentence neighborhoods, enabling more meaningful local analysis.

2. **Post-run Phase: ILL Feature Extraction**  
   A comprehensive feature extraction pipeline for analyzing loss landscapes and neighborhood properties, adaptable to any neighborhood definition.

3. **Flexible and Modular Evaluation**  
   The phases can be used separately or together, supporting a wide range of benchmarking, classification, and exploratory analysis tasks.

**Research Goals:**
- To introduce and benchmark the two-phase framework for unlearning analysis.
- To evaluate the effectiveness of each phase, individually and in combination, for distinguishing between forget, retain, and holdout subsets.
- To provide a flexible toolkit that can be adapted for future research in model evaluation and unlearning.

*This work advances the methodology for unlearning analysis by providing modular, extensible tools that can serve both as benchmarks and as investigative instruments for uncovering new phenomena in model behavior.*

# Histograms

## hist_original_loss 

The retain group’s losses are tightly concentrated at low values, with a steep right skew and minimal spread. The holdout group peaks at higher values, with a wider and moderately right-skewed distribution. The forget group sits between the two, with its peak closer to the retain, overlapping more with retain than with holdout. Overlap between retain and holdout is almost non-existent.  

- The sharp separation and minimal overlap between retain and holdout indicate strong domain boundaries. **This could reflect overfitting on the retain set and limited generalization to unseen data.**  
- The forget group’s position closer to retain indicates that, before unlearning, it was learned almost as well as the retain set. **That means forgetting will likely require substantial loss increases to match holdout difficulty.**  
- The wider spread of the holdout group suggests greater variability in difficulty for unseen data. **This makes it a valuable benchmark for detecting over- or under-forgetting after unlearning.**

## hist_mean_neighbor_loss  

The retain group has the lowest mean neighbor loss, with a narrow, right-skewed distribution. The forget group peaks in the middle, overlapping substantially with both retain and holdout. The holdout group has the highest mean neighbor loss, with a wide, right-skewed distribution.  

- The retain group’s low mean neighbor loss suggests that the sentences the model should keep are surrounded by other sentences that are also well-understood by the model. **This implies that the retain data forms a dense, well-defined cluster in the model’s feature space, which is critical for ensuring that unlearning the forget set doesn’t degrade performance on the core task.**  
- The forget group’s middle position and high overlap with both other sets mean its neighbors vary in difficulty. **This could signal partial forgetting — some regions shifted towards holdout difficulty while others remain closer to retain.**  
- The holdout group’s high mean neighbor loss and wide distribution indicate that unseen data points are surrounded by a more diverse range of sentences, many of which the model finds difficult. **This suggests that the model’s feature space for unseen data is less structured and more spread out.**  
- The significant overlap of the forget group’s distribution with both retain and holdout distributions suggests that the sentences to be forgotten are not entirely unique in the model’s feature space. **This raises a warning: the information to be unlearned may be intertwined with both retain and holdout data, meaning that simply removing the forget data could unintentionally degrade performance on the other sets — an assumption that should be explicitly tested.**

## hist_max_neighbor_loss  

The retain group has the lowest maximum neighbor loss, featuring a narrow, right-skewed distribution. The forget group occupies the middle range, overlapping substantially with both retain and holdout. The holdout group shows the highest maximum neighbor loss with the widest distribution, indicating a large spread and moderate overlap with retain.  

- The retain group has the lowest maximum neighbor loss, suggesting that even the most difficult sentences in the neighborhood of the retain data are still relatively well-understood by the model. **This reinforces the idea that the core knowledge the model needs to preserve is in a stable, well-learned region of the feature space.**  
- The forget group’s intermediate position and overlap imply that some forgotten examples have neighbors as difficult as those in holdout, while others are closer to retain. **This heterogeneity indicates unlearning challenges where some forget data is deeply embedded near critical knowledge regions.**  
- The holdout group’s high maximum neighbor loss and wide spread indicate that many unseen data points have at least one highly difficult neighbor. **This suggests that the model’s feature space for unseen data is not only less structured but also contains more "outlier" or challenging examples, which is a key characteristic to consider when evaluating the unlearning process.**  
- The significant overlap of forget with both retain and holdout warns that maximum neighbor losses alone may not clearly separate forgotten from retained information. **This calls for careful evaluation to avoid inadvertent forgetting or overfitting.**

## hist_min_neighbor_loss  

The retain group has the lowest minimum neighbor loss, with a narrow, left-skewed distribution. The forget group sits in the middle, overlapping substantially with retain and partially with holdout. The holdout group shows the highest minimum neighbor loss, with a wider distribution and a peak shifted to the right.  

- The retain group’s very low minimum neighbor losses suggest that each retain example has at least one nearby sentence the model understands extremely well. **This indicates a dense, supportive local feature space that helps preserve core task performance.**  
- The forget group’s middle position and overlap with both retain and holdout imply that some forgotten examples still have very easy neighbors, while others are closer in difficulty to holdout. **This mix suggests partial forgetting and uneven separation in the feature space.**  
- The holdout group’s higher minimum neighbor losses and wider spread indicate that even the easiest neighbors for unseen data are relatively challenging compared to retain and forget. **This points to less structured and less supportive neighborhoods for generalization.**  
- **WARNING:** It will be interesting to check correlation between the original loss (OL) and the min_neighbor_loss (MNL). samples with lower OL had lower MNL? are some MNL from houldout lower then OL in retain? and so on.

# Neighbors min/max/mean loss summary: Insights  

- We see that the three loss manifolds (holdout, forget, retain) are stuck one on top of the other.
- The high values in the min_neighbor_loss of retain (which are around 2.0-2.5) and forget (~2.6) are lower than the lowest values in the max_neighbor_loss of holdout (which are around 3.0). Indicating a clear separation between the holdout group and the others in terms of neighborhood loss range.

## hist_loss_variance  

All three groups have right-skewed distributions with most values concentrated near zero, indicating generally consistent model performance across neighbors. The holdout group has the highest concentration of low variance, peaking on the far left. The retain group’s distribution is slightly wider with a lower peak, while the forget group’s peak is similar in height but shifted slightly right, reflecting a higher average variance. Overlap among all three groups is very high.  

- The holdout group’s very low variance suggests that unseen examples tend to have neighbors of similar difficulty. **This stability may reflect the model’s tendency to generalize uniformly in regions of the feature space it has not explicitly trained on.**  
- The retain group’s slightly higher variance indicates more variability in neighbor difficulty. **This could stem from the retain set containing a mix of both very easy and moderately challenging examples.**  
- The forget group’s right-shifted peak shows it experiences the highest average variance among the three. **This may imply that forgetting disrupts local neighborhood consistency, causing some neighbors to become substantially harder while others remain easy.**  
- The high overlap of all three distributions means variance alone is unlikely to distinguish between forget, retain, and holdout. **This limits its discriminative power for evaluating unlearning effects and suggests the need for complementary metrics.**

## hist_loss_std 
All three groups have right-skewed distributions, with most values showing low standard deviation. This suggests that losses within neighborhoods are generally consistent. The holdout group has the lowest standard deviations, peaking furthest to the left. The retain group is in the middle, and the forget group is shifted to the right, meaning it has more variation. Despite these differences, the three groups overlap a lot.

The holdout group has the highest concentration of low standard deviation, peaking on the far left. The forget group sits in the middle, while the retain group is shifted to the right, reflecting a higher average standard deviation. Overlap among all three groups is very high.  

- The holdout group’s very low standard deviations suggest that unseen examples typically have neighbors with similar losses. **This points to stable but possibly uniformly challenging neighborhoods in the feature space for unseen data.** 
- The low standard deviation in the holdout group suggests that its local loss landscapes are smoother and more uniform. This can indicate stable generalization for unseen data.
- The higher variation in the forget group points to more irregular and less predictable local loss landscapes after unlearning, suggesting that the model’s representation in those regions was altered more substantially.
- The retain group’s intermediate variation suggests it maintains a mix of stable and more varied regions, potentially reflecting preserved decision boundaries with some local irregularities.
- The high overlap between groups implies that, despite shifts in variation, many regions of the input space retain similar local smoothness properties, meaning unlearning affects only certain subregions rather than globally altering the landscape.
- The strong overlap between all three groups means standard deviation alone is unlikely to reliably separate forget, retain, and holdout. **This underscores the need for combining it with other neighborhood metrics when evaluating unlearning effects.**

## hist_mean_loss_increment  

All three groups show right-skewed distributions of mean loss increments between sentences and their neighborhoods, with peaks at low values. The holdout group has the highest peak at a low increment, followed closely by retain. The forget group’s distribution is more spread out, with its peak slightly left-shifted and a long tail toward the highest increments.  

- The holdout group’s sharp peak at low mean loss increments indicates that sentences in unseen data tend to have neighbors with very similar losses. **This suggests local smoothness and stable generalization in the model’s feature space for unseen regions.**  
- The retain group’s similar but slightly lower peak shows that sentences to be retained also generally reside in neighborhoods with minimal loss change. **This reflects stable knowledge clusters preserved during unlearning.**  
- The forget group’s wider distribution and long tail of large increments mean that for some sentences, neighbors differ substantially in loss. **This reveals that forgetting creates irregular local changes, with some neighbors becoming much harder, reflecting uneven unlearning.**  
- The overlap between groups and presence of large increments outside forget imply that loss increment changes are not fully unique to forgetting. **This warns that unlearning effects might unintentionally propagate beyond the forget set, requiring careful evaluation.**

## hist_max_loss_increment  

The maximum loss increment distributions for all three groups are right-skewed with peaks clustered at similarly low values. The holdout group has the highest peak at a low increment, followed by retain. The forget group exhibits the longest tail, while the retain group has a wider spread despite a shorter tail.  

- The holdout and forget groups' high peak at low maximum loss increments suggests that most unseen and forget sentences have neighbors whose losses change very little, indicating stable local neighborhoods. **This reflects consistent generalization in unseen feature space regions, and this reflects consistent generalization-level in unseen feature space regions, and suggests that unlearning reducing local maxima around the forget sentence.**  
- The retain group’s wider spread but shorter tail indicates more frequent moderate increments among neighbors, but fewer extreme changes. **This could imply subtle variations in how retaining sentences influence neighborhood loss during unlearning.**  
- The forget group’s longest tail shows that some forgotten sentences have neighbors with very large increments in loss. **This highlights effective forgetting for certain difficult neighbors, but also points to uneven local impact within the forget subset.**  

## hist_min_loss_increment  

The minimum loss increment distributions show the holdout group with a sharp peak near 0.5 and a relatively narrow, right-skewed shape. The retain group has a similar peak but with a slightly wider spread. The forget group is the most spread out, featuring a lower peak and a long tail extending into both negative and very high positive increments.  

- The holdout group’s narrow peak near 0.5 indicates that, for unseen sentences, the smallest neighborhood loss increments are generally moderate and consistent. **This suggests a stable but uniformly challenging local neighborhood structure in unseen data.**  
- The retain group’s slightly wider distribution implies more variability in minimal loss increments among neighbors, though still centered near 0.5. **This reflects some heterogeneity in how retained sentences relate to their closest neighbors during unlearning.**  
- The forget group’s broad spread and tail into negative increments show that some forgotten sentences have neighbors whose losses actually decreased, while others saw very large increases. **This points to uneven unlearning effects, including possible unintended local improvements or noisy fluctuations.**  

# Increment hists summary: Insights - Impact of Unlearning on Increment Distributions
 
The unlearning process reduces the **max**, **min**, and **mean** increment values for the *forget* set, effectively shifting them away from their pre-unlearning (retain-like) range toward values more aligned with the *holdout* set.  

- Lower max, min, and mean increments after unlearning suggest that the loss surface around forget samples has become *flatter*. The model’s predictions are now less sensitive to small perturbations in these samples, similar to the holdout set.

- It implies that the model is treating the forgotten data more like unseen data (holdout) rather than memorized (retain) data.  

- A drop in maximum increments means there are fewer sharp loss spikes around forget samples. This hints at the removal of highly specialized decision boundaries that were tailored to them, erasing fine-grained memorization.

# Together with neighbors min/max/mean loss:

We saw that the three loss manifolds (holdout, forget, retain) are stuck one on top of the other. The increment features tells us that if before, the forget was part of the retain, now it's manifold is squeezed and pushed upwards: above the retain but lower then the holdout (act as an upper boundary). I assume that above the boundary the model starts to lose performance, that make sense because our goal it to keep the holdout (test) performance as close as possible to the train (retain) performance. As a result, the forget group ends up squeezed in between them.

## hist_mean_gradient  

The plot shows the distribution of mean gradients (local loss landscape steepness) for the forget, retain, and holdout groups. The holdout group has the highest concentration of low mean gradient values, indicating a smoother local loss landscape. The forget group is similar but with lower peak, its lines merge with those of the holdout. The retain group has a peak at the of forget, but its distribution is shifted a bit to the right. The forget and holdout are highly overlapping and with some (~75%) overlap with retain

- The holdout group’s concentration at low mean gradients suggests that unseen sentences are surrounded by neighbors with similar losses relative to their distance, indicating smooth, stable regions in the feature space. **This points to well-behaved generalization on unseen data.**
- The forget group’s strong overlap with holdout means the model’s loss landscape near forget data also became smoother after unlearning, aligning those regions more with unseen data. 
- The retain group’s shift toward higher mean gradients suggests more complex or steeper loss landscapes near retained sentences, reflecting actively learned decision boundaries.  

## hist_max_gradient

This histogram shows the distribution of the maximum gradient loss values for the forget, retain, and holdout groups. The forget and holdout groups have very similar distributions, both peaking sharply at lower gradient values, indicating many points with relatively small maximum local steepness. The retain group’s distribution is wider and skewed toward higher maximum gradients, showing a greater number of points with steep local loss landscapes.

**Insights:**
- The retain group’s wider distribution and higher concentration of large max gradients suggest that some retained sentences are located in more unstable or complex regions of the loss landscape, likely reflecting difficult or influential examples actively learned by the model.   
- The holdout group’s tight clustering at low gradient values indicates mostly smooth local loss landscapes in unseen data. **This stability may facilitate consistent generalization in feature space regions not directly affected by training or forgetting.**
- The fact that the forget group’s max gradients do not show higher peaks indicates the forget data is not inherently “harder” or more complex for the model than holdout data.

## hist_gradient_variance  

The histogram shows all three groups with a strong peak near zero and right-skewed distributions. All with very long narrow right skew

- The peak near zero across all groups suggests most examples have stable, consistent local gradient behavior, with minimal variation in gradient steepness across neighbors, reflecting model confidence and consistent behavior. **This indicates generally smooth loss landscapes in the neighborhood for most data points.**   
- The long right tails suggest that a minority of examples in each group experience significant variability in local gradients, highlighting the presence of outlier points or regions of unstable learning.  
- This pattern implies that while most of the model’s feature space is well-learned and stable, certain instances—possibly difficult or ambiguous sentences—cause localized fluctuations in loss landscape steepness

## hist_loss_volatility (Fluctuation, Instability)

The histogram shows the distribution of loss volatility across the three subsets. The holdout group has a sharp peak at very low volatility, indicating stable loss behavior. In contrast, the forget and retain groups have wider, right-skewed distributions, with peaks at higher volatility values. The forget group is slightly more concentrated toward lower volatility compared to retain (between the holdout to retain).  

- The holdout group’s low volatility peak suggests that the model’s predictions for unseen data are stable and robust to small perturbations, indicative of a smooth input loss landscape.  
- The wider and right-skewed distributions for the forget and retain groups imply these training examples reside in more volatile regions, where loss fluctuates more with input variations, making them inherently more difficult or sensitive to learn and unlearn.
- The forget group’s intermediate volatility concentration suggests it occupies a transitional space, more stable than retain but less so than holdout. Highlighting that the unlearning process may need to carefully manage this instability to avoid unintended effects on the retain data.

## hist_increment_variance  

This plot shows the distribution of variance in loss increments across the three data subsets. All three distributions are right-skewed with sharp peaks near zero. 
Although very subtle, the holdout peak is higher and more to the left than the retain and forget, and the forget is higher and more to the left than retain. but the overlap is very big. The retain right skew is a bit less sharp.

- The high concentration near zero variance across all groups suggests that for most data points, the loss increments are stable and predictable when the model's inputs or weights are perturbed, indicating a generally smooth local loss landscape.  
- The subtle leftward shift of the holdout peak implies that unseen data points experience slightly more consistent loss increments, reinforcing their role as stable benchmarks.  
- The retain group’s broader and less sharply right-skewed distribution indicates a small subset of examples with more volatile loss increments, suggesting these points are more sensitive to perturbations and potentially more vulnerable to collateral effects during unlearning.  
- The forget group's intermediate position in this distribution highlights that its loss increment variability is between the stable holdout and more volatile retain sets, reflecting its transitional nature in the input loss landscape and the complexity of isolating its influence during unlearning.  


# Classifier Performance Comparison plot - Multi-Class

This set of three bar charts compares classification performance across three tasks—multi-class, Holdout vs All, and Forget vs All—using Accuracy, F1 Score, and ROC AUC metrics. Overall, the Holdout vs All task consistently achieves the highest scores, multi-class is intermediate, and Forget vs All is the most challenging across all metrics.

| Metric       | Logistic Regression | Random Forest |
|--------------|---------------------|---------------|
| Accuracy     | 0.779               | 0.742         |
| F1 Score     | 0.772               | 0.740         |
| ROC AUC      | 0.892               | 0.890         |

- The high performance pattern on Holdout vs All, intermediate on multi-class, and lowest on Forget vs All confirms that the holdout set is the most distinct, the forget set is the hardest to isolate, and the retain and forget sets share substantial feature overlap.
- Logistic Regression slightly outperforms or matches Random Forest on the multi-class and Holdout vs All tasks, indicating that the key features distinguishing these datasets are largely linearly separable or effectively captured by a linear model.  
- The Forget vs All task shows a stark contrast, with Random Forest significantly outperforming Logistic Regression, especially in F1 score (Random Forest ~0.6 vs Logistic Regression close to 0.1), suggesting strong non-linearities in distinguishing forget data that linear models fail to capture.  
- The relatively high accuracy but low F1 on Forget vs All for Logistic Regression highlights the difficulty in correctly identifying forget instances, pointing to subtle overlap rather than outright separability, which poses a challenge for unlearning algorithms.

## more information later

# Confusion Matrix Analysis - Multi Class

## Logistic Regression  

The logistic regression classifier’s performance across the three classes—retain, holdout, and forget—is summarized below:

| Class   | Precision | Recall | F1-Score | Support |
|---------|-----------|--------|----------|---------|
| Retain  | 0.74      | 0.81   | 0.78     | 189     |
| Holdout | 0.86      | 0.94   | 0.89     | 189     |
| Forget  | 0.71      | 0.57   | 0.63     | 189     |

Overall accuracy is 0.77, with balanced macro and weighted averages.

**Insights:**  
- **Holdout data** is well-separated and easy to classify, confirming its role as a clean benchmark.  
- **Retain data** has good recall but lower precision, indicating some overlap with other classes and potential classification ambiguity.  
- **Forget data** shows the lowest recall and F1, reflecting difficulty isolating it due to overlap with retain and holdout, highlighting challenges for unlearning.   

## Classification Report - Random Forest  

| Class   | Precision | Recall | F1-Score | Support |
|---------|-----------|--------|----------|---------|
| Retain  | 0.88      | 0.88   | 0.88     | 189     |
| Holdout | 0.96      | 0.97   | 0.97     | 189     |
| Forget  | 0.87      | 0.85   | 0.86     | 189     |

**Overall accuracy:** 0.90  

---

### Insights

- The **Random Forest** significantly improves classification across all classes compared to Logistic Regression, indicating complex, non-linear feature relationships.  
- **Holdout** remains the easiest to classify, reinforcing its distinctness and reliability as a benchmark.  
- The **forget set** classification improves notably, yet still lags slightly behind retain and holdout, reflecting persistent overlap and unlearning difficulty.   


### RF Confusion Matrix

The confusion matrix summarizes the Random Forest classifier’s ability to distinguish between retain, holdout, and forget data subsets based on extracted features. Correct classifications are shown on the diagonal: 167 for retain, 184 for holdout, and 161 for forget. Misclassifications appear off-diagonal, revealing specific confusion patterns.  

| Actual  | Predicted Retain | Predicted Holdout | Predicted Forget |
|---------|------------------|-------------------|------------------|
| Retain  | 167              | 2                 | 20               |
| Holdout | 0                | 184               | 5                |
| Forget  | 23               | 5                 | 161              |

- The model classifies holdout data with the highest accuracy (184 correct, only 5 confused as forget and none as retain), suggesting that holdout examples have more distinctive or consistent features.  
- Retain data is mostly well-classified (167 correct), but there is notable confusion with forget (20 misclassified as forget), implying overlap or feature similarity between retain and forget subsets.  
- Forget data is more often misclassified as retain (23 instances) than as holdout (5 instances), indicating some forget examples share features with retain, making them harder to isolate.  
- The confusion between retain and forget highlights the challenge in clearly separating these subsets, which may impact unlearning effectiveness if features are not sufficiently discriminative.  
- Overall, the model performs well but reveals subtle overlaps between retain and forget classes, underscoring the importance of feature design and selection in unlearning research.

# Multi-class ROC Curves — Random Forest  

The ROC curves for the Random Forest classifier show excellent discrimination between the forget, retain, and holdout classes. All curves hug the top-left corner, reflecting very high true positive rates and low false positive rates across thresholds. The Area Under the Curve (AUC) values range from 0.95 to 0.99, indicating near-perfect classification performance for all classes.  

- The high AUC scores confirm that features extracted from the data—such as loss volatility, increment variance, and gradients—are extremely effective at distinguishing between forget, retain, and holdout data points. **This validates the feature engineering approach of the research project, enabling a high-performing classifier.**  
- The holdout class has the highest AUC of 0.99, showing the model can almost perfectly distinguish unseen data from the other two subsets. **This suggests the holdout data is feature-wise very distinct, making it an excellent benchmark for evaluating unlearning without contamination.**  
- The forget and retain classes have slightly lower but still excellent AUCs of 0.95 and 0.97, respectively. **This aligns with the confusion matrix and histogram analyses showing some overlap and similarity between these groups, highlighting a key challenge in machine unlearning to clearly separate them.**  
- The strong ROC performance underscores the quality of the extracted features but also points to the subtlety required in improving separation between forget and retain classes for more precise unlearning.

# Comparative Analysis of ROC Curves  

This plot compares ROC curves for two classifiers—Random Forest and Logistic Regression—on binary tasks distinguishing a specific data subset (Forget or Holdout) from all other data.  

- **Random Forest consistently outperforms Logistic Regression**, especially on the Forget vs All task (AUC 0.821 vs. 0.712), highlighting its strength in capturing non-linear relationships between features and target classes.  
- For the Holdout vs All task, both models perform exceptionally well with nearly identical AUCs (Random Forest 0.964, Logistic Regression 0.963), confirming that holdout data is highly distinct and serves as a clean benchmark for unlearning evaluation.  
- The Forget vs All task is the most challenging, reflected by the lowest AUC scores. The Random Forest’s better performance here indicates it handles the subtle and overlapping feature space between forget and other data better than Logistic Regression.  
- These results align with prior confusion matrix and histogram analyses showing significant overlap between forget and retain data, emphasizing the difficulty in isolating forget examples and the importance of sophisticated modeling approaches.  
- Overall, the strong performance on holdout separation versus the relative difficulty on forget separation underscores the nuanced challenges in effective machine unlearning.  

- **Model Performance:** The Random Forest classifier is better suited for this feature-based classification task, likely due to the non-linear nature of the relationships between features like loss volatility and gradient variance.  
- **Distinctness of Data Subsets:** The holdout set is highly distinct and easily separable from the other data, confirming its role as a valid benchmark. The forget and retain sets, however, show more feature-space overlap, making the forget set harder to isolate and potentially harder to unlearn without impacting the retain data.  
- **Unlearning Implications:** The difficulty in isolating the forget set with a simple classifier suggests that a machine unlearning algorithm would also face challenges. The feature space overlap means that unlearning methods targeting the forget data may inadvertently affect similar retain data, potentially degrading the model’s performance on information it should retain.

# Classifier Performance Comparison plot

This set of three bar charts compares the performance of two classifiers, Logistic Regression and Random Forest, across three different tasks: multi-class classification and two binary classification tasks (Holdout vs All and Forget vs All). The performance is measured by Accuracy, F1 Score, and ROC AUC.

**Description:**  
The Random Forest classifier consistently outperforms Logistic Regression, particularly in the multi-class and Forget vs All tasks. This suggests that the features used to train these classifiers have complex, non-linear relationships that the Random Forest model is better at capturing. The multi-class accuracy for Random Forest is approximately 0.75, compared to Logistic Regression's 0.65, highlighting its superior ability to distinguish between all three data subsets.

Both classifiers are extremely effective at the Holdout vs All task, achieving near-perfect scores across all metrics (F1 and ROC AUC are both above 0.95). This confirms that the holdout dataset is distinctly different from the other data subsets, making it an excellent benchmark for measuring the impact of the unlearning process without contamination.

The Forget vs All task is the most challenging for both classifiers, with the lowest scores across all metrics. The Logistic Regression model performs particularly poorly, with an F1 score close to 0.1, indicating its difficulty in isolating the forget data. The Random Forest model, while better, still shows its lowest performance on this task (F1 score of ~0.6), which suggests that the forget and retain datasets have a significant degree of overlap in their feature space, making it difficult to distinguish them.

**Insights:**
- Random Forest is consistently superior to Logistic Regression, especially for tasks with complex decision boundaries.
- The Holdout vs All task is trivially easy for both models, confirming the holdout set’s distinctiveness.
- Forget vs All is the hardest task, with substantial overlap between forget and retain data leading to reduced classification performance.
- The large performance gap between models on Forget vs All suggests that model choice significantly impacts unlearning evaluation results.

# PCA and t-SNE Embeddings  

This plot displays two-dimensional embeddings of retain, holdout, and forget data subsets. The PCA plot on the left shows considerable overlap between retain and forget groups, while the holdout group is somewhat distinct. The t-SNE plot on the right reveals much clearer separation, with holdout forming a distinct cluster and retain and forget groups also showing high degrees of separation.  

- The PCA plot’s significant overlap between retain and forget, especially along the first principal component, indicates that a linear transformation is insufficient to separate these subsets. **This aligns with ROC and confusion matrix findings, suggesting simple unlearning methods may struggle to isolate forget data without affecting retain data.**  
- The t-SNE plot’s clear and distinct clusters for holdout, retain, and forget highlight the presence of non-linear feature relationships that separate these groups more effectively. **This confirms the holdout data’s independence and supports the validity of non-linear feature representations.**  
- The difference between PCA and t-SNE separation underscores the non-linear nature of the distinguishing features. **This validates the use of non-linear classifiers such as Random Forest and implies that successful unlearning methods will need to model complex, non-linear dependencies in the data.**  
- The distinct t-SNE clusters for retain and forget suggest that while challenging, it is possible to differentiate these groups with appropriate non-linear embedding and modeling techniques, guiding the design of more precise unlearning algorithms.

# Isomap and UMAP Embeddings  

This plot visualizes the retain, holdout, and forget data subsets using two non-linear dimensionality reduction methods: Isomap and UMAP. The Isomap plot shows significant overlap between retain and forget groups with only partial separation of holdout, whereas the UMAP plot reveals clear, distinct clusters for all three groups, indicating strong non-linear structure in the feature space.  

- The Isomap plot’s overlap of retain and forget suggests that a simpler non-linear embedding does not fully disentangle these subsets, implying their feature characteristics are closely intertwined and challenging to separate.  
- The UMAP plot successfully isolates the holdout group, reaffirming that its features are fundamentally distinct and validating its role as a reliable, unbiased benchmark in unlearning evaluation.  
- The clear separation of all three groups by UMAP, a more powerful non-linear technique, shows that while retain and forget are close in some feature dimensions, they are distinguishable when complex non-linear relationships are preserved.  
- This finding is crucial for machine unlearning: it demonstrates that the forget set is not identical to retain and can theoretically be isolated and unlearned, though the subtle non-linear distinctions suggest that unlearning will require sophisticated, nuanced approaches.

# Cross-Dataset Prediction Transfer Analysis  

The function `analyze_prediction_transfer` trains logistic regression models on one dataset subset (retain, holdout, or forget) versus all others, then tests the model’s generalization on a different subset versus its complements. The resulting accuracy and AUC scores measure how well features and learned decision boundaries transfer across data splits.  

**Cross-Dataset Transfer Accuracy (text heatmap):**  
| Train  | Test Retain | Test Holdout | Test Forget |  
|--------|-------------|--------------|-------------|  
| Retain | 0.318       | 0.501        | —           |  
| Holdout| 0.323       | 0.401        | —           |  
| Forget | 0.653       | 0.604        | —           |  

**Cross-Dataset Transfer AUC (text heatmap):**  
| Train  | Test Retain | Test Holdout | Test Forget |  
|--------|-------------|--------------|-------------|  
| Retain | 0.039       | 0.553        | —           |  
| Holdout| 0.111       | 0.420        | —           |  
| Forget | 0.643       | 0.145        | —           |  

---

- The models trained on the **forget** subset generalize substantially better to both retain and holdout test sets compared to models trained on retain or holdout. This is evident from the higher accuracy (0.653 and 0.604) and AUC (0.643 and 0.145) scores, especially notable on test retain data.  
- Models trained on **retain** or **holdout** subsets exhibit poor transfer performance to other subsets, with very low AUCs (e.g., retain→retain AUC 0.039, holdout→retain 0.111), suggesting limited cross-subset generalization from these features.  
- The low AUC and accuracy scores overall indicate that binary classifiers trained on one subset struggle to predict other subsets, highlighting significant feature distribution differences and limited transferability.  
- This asymmetry in transfer performance may suggest that forget data contains more discriminative or distinctive features that overlap with retain and holdout, while retain and holdout are less predictive of forget, consistent with prior overlap analyses.  
- These results underscore challenges in unlearning: models specialized to one subset may fail to generalize, complicating efforts to robustly separate and remove forget data without unintended impacts.  

**Summary:**  
The cross-dataset transfer results reveal that forget-trained models have relatively better generalization across data splits, but the overall low AUC and accuracy values emphasize substantial divergence in feature distributions. This reinforces the complexity of isolating forget data for effective machine unlearning, requiring methods that can handle these nuanced feature relationships and distributional shifts.

### Additional Insights and Implications for Machine Unlearning

- **Forget Set’s Informative Nature:** The forget subset contains features that generalize to retain and holdout sets, implying that the model’s knowledge about forget data is intertwined with other subsets. This interconnectedness complicates complete unlearning, as naïve removal of forget data risks degrading model performance on retain and holdout data.  

- **Distinctness of Retain and Holdout Features:** The poor transfer from retain and holdout to other subsets suggests these sets have unique or less representative features. This reinforces the holdout’s role as a reliable benchmark and indicates that the retain set’s features must be carefully preserved during unlearning to avoid unintended loss of important knowledge.  

- **Challenge for Unlearning Algorithms:** The asymmetric transferability highlights the nuanced complexity of machine unlearning. Effective algorithms need to surgically remove forget-related knowledge that is entangled with retain and holdout information, rather than broadly erasing overlapping features. This demands sophisticated methods capable of disentangling subtle, non-linear feature relationships to minimize collateral damage.  


# How much the original sentence is important?

It will be interesting to check the performance of the method on neighbors of the original sentence. In the real-world settings won't have the exact sentence and we will run the method on its neighbor, and therefore will get a shifted neighborhood, how will it affect the results?

I haven't run it yet.